In [1]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output


# 1. ESTRUCTURA DEL ÁRBOL (BST)

class NodoCancion:
    def __init__(self, tiempo, titulo="Desconocido"):
        self.tiempo = tiempo      
        self.titulo = titulo      
        self.izq = None           
        self.der = None          

class Playlist:
    def __init__(self):
        self.raiz = None

    def insertar(self, tiempo, titulo):
        nuevo = NodoCancion(tiempo, titulo)
        if self.raiz is None:
            self.raiz = nuevo
            return
        actual = self.raiz
        while True:
            if tiempo < actual.tiempo:
                if actual.izq is None:
                    actual.izq = nuevo
                    break
                actual = actual.izq
            elif tiempo > actual.tiempo:
                if actual.der is None:
                    actual.der = nuevo
                    break
                actual = actual.der
            else:
                break

   
    def eliminar_nodo(self, raiz, tiempo):
        if raiz is None:
            return raiz
        
        if tiempo < raiz.tiempo:
            raiz.izq = self.eliminar_nodo(raiz.izq, tiempo)
        elif tiempo > raiz.tiempo:
            raiz.der = self.eliminar_nodo(raiz.der, tiempo)
        else:
            # Casos con 0 o 1 hijo
            if raiz.izq is None:
                temp = raiz.der
                raiz = None
                return temp
            elif raiz.der is None:
                temp = raiz.izq
                raiz = None
                return temp
            
         
            temp = self._min_valor_nodo(raiz.der)
            raiz.tiempo = temp.tiempo
            raiz.titulo = temp.titulo
            raiz.der = self.eliminar_nodo(raiz.der, temp.tiempo)
            
        return raiz

    def _min_valor_nodo(self, nodo):
        actual = nodo
        while actual.izq is not None:
            actual = actual.izq
        return actual

 
    def calcular_tiempo_total(self, nodo_actual):
        if nodo_actual is None:
            return 0
        return (nodo_actual.tiempo + 
                self.calcular_tiempo_total(nodo_actual.izq) + 
                self.calcular_tiempo_total(nodo_actual.der))

 
    def recomendar_cancion_perfecta(self, segundos):
        if self.raiz is None:
            return None
        actual = self.raiz
        mejor_nodo = actual
        menor_dif = abs(actual.tiempo - segundos)

        while actual is not None:
            dif_actual = abs(actual.tiempo - segundos)
            if dif_actual < menor_dif:
                menor_dif = dif_actual
                mejor_nodo = actual
            if dif_actual == 0:
                break
            if segundos < actual.tiempo:
                actual = actual.izq
            else:
                actual = actual.der
        return mejor_nodo

  
    def filtrar_canciones_cortas(self, tiempo_min):
        self.raiz = self._filtrar_recursivo(self.raiz, tiempo_min)

    def _filtrar_recursivo(self, nodo, tiempo_min):
        if nodo is None:
            return None
        nodo.izq = self._filtrar_recursivo(nodo.izq, tiempo_min)
        nodo.der = self._filtrar_recursivo(nodo.der, tiempo_min)
        if nodo.tiempo < tiempo_min:
            return nodo.der
        return nodo


    def contar_canciones(self, nodo):
        if nodo is None:
            return 0
        return 1 + self.contar_canciones(nodo.izq) + self.contar_canciones(nodo.der)

    def obtener_promedio(self):
        total_tiempo = self.calcular_tiempo_total(self.raiz)
        total_canciones = self.contar_canciones(self.raiz)
        if total_canciones == 0:
            return 0
        return total_tiempo / total_canciones

   
    def obtener_lista_ordenada(self, nodo, lista):
        if nodo:
            self.obtener_lista_ordenada(nodo.izq, lista)
            lista.append(nodo.tiempo)
            self.obtener_lista_ordenada(nodo.der, lista)

    # Coordenadas para visualización Gráfica
    def _calcular_coordenadas(self, nodo, x=0, y=0, espacio=10, data=None):
        if data is None: data = []
        if nodo:
            data.append((x, y, f"{nodo.tiempo}s"))
            if nodo.izq:
                data.append(((x, y), (x - espacio, y - 1)))
                self._calcular_coordenadas(nodo.izq, x - espacio, y - 1, espacio * 0.6, data)
            if nodo.der:
                data.append(((x, y), (x + espacio, y - 1)))
                self._calcular_coordenadas(nodo.der, x + espacio, y - 1, espacio * 0.6, data)
        return data

# 2. INTERFAZ GRÁFICA (Estilo App de Escritorio)

playlist = Playlist()
salida_consola = widgets.Output()
salida_grafica = widgets.Output()

# --- Funciones de actualización visual ---
def actualizar_estado_arbol(mensaje_extra=""):
    with salida_consola:
        clear_output()
        print("\033[40m\033[37m", end="") 
        
        lista_tiempos = []
        playlist.obtener_lista_ordenada(playlist.raiz, lista_tiempos)
        
        print("Duración de canciones (orden):")
        print(f"{lista_tiempos}\n")
        
        total_canciones = playlist.contar_canciones(playlist.raiz)
        tiempo_total = playlist.calcular_tiempo_total(playlist.raiz)
        promedio = playlist.obtener_promedio()
        
        print(f"Total canciones: {total_canciones}")
        print(f"Tiempo total: {tiempo_total} segundos")
        print(f"Promedio: {promedio:.2f} segundos\n")
        
        if mensaje_extra:
            print(f">>> {mensaje_extra}")
            
def dibujar_arbol():
    with salida_grafica:
        clear_output()
        if playlist.raiz is None:
            return
        elementos = playlist._calcular_coordenadas(playlist.raiz)
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.axis('off')
        for elem in elementos:
            if len(elem) == 2:
                p1, p2 = elem
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='black', zorder=1)
        for elem in elementos:
            if len(elem) != 2:
                x, y, texto = elem
                ax.scatter(x, y, color='#4CAF50', s=1000, zorder=2)
                ax.text(x, y, texto, color='white', ha='center', va='center', fontweight='bold', zorder=3)
        plt.tight_layout()
        plt.show()

# --- Controladores de Botones ---
def btn1_calcular_click(b):
    actualizar_estado_arbol("Operación 1: Tiempo total calculado con éxito.")

def btn2_recomendar_click(b):
    try:
        seg = int(in_recomendar.value)
        nodo = playlist.recomendar_cancion_perfecta(seg)
        if nodo:
            msg = f"Operación 2: Canción recomendada -> {nodo.titulo} ({nodo.tiempo}s)"
        else:
            msg = "Árbol vacío. No hay recomendaciones."
        actualizar_estado_arbol(msg)
        in_recomendar.value = ""
    except ValueError: pass

def btn3_filtrar_click(b):
    try:
        minimo = int(in_filtrar.value)
        playlist.filtrar_canciones_cortas(minimo)
        actualizar_estado_arbol(f"Operación 3: Canciones menores a {minimo}s eliminadas.")
        dibujar_arbol()
        in_filtrar.value = ""
    except ValueError: pass

def btn4_promedio_click(b):
    actualizar_estado_arbol("Operación 4: Promedio calculado (ver arriba).")

def btn5_insertar_click(b):
    try:
        tiempo = int(in_insertar_tiempo.value)
        titulo = in_insertar_titulo.value.strip() or f"Track_{tiempo}s"
        playlist.insertar(tiempo, titulo)
        actualizar_estado_arbol(f"Operación 5: Insertada nueva canción de {tiempo}s.")
        dibujar_arbol()
        in_insertar_tiempo.value = ""
        in_insertar_titulo.value = ""
    except ValueError: pass

def btn_eliminar_click(b):
    try:
        tiempo = int(in_eliminar.value)
        playlist.raiz = playlist.eliminar_nodo(playlist.raiz, tiempo)
        actualizar_estado_arbol(f"Mantenimiento: Canción de {tiempo}s eliminada del sistema.")
        dibujar_arbol()
        in_eliminar.value = ""
    except ValueError: pass

def btn6_reiniciar_click(b):
    playlist.raiz = None
    datos_ejemplo = [250, 170, 314, 110, 210, 324, 211]
    for d in datos_ejemplo:
        playlist.insertar(d, f"Canción {d}s")
    actualizar_estado_arbol("Operación 6: Árbol reiniciado con datos de ejemplo.")
    dibujar_arbol()

# --- Diseño de la Interfaz ---
estilo_btn = widgets.Layout(width='95%', height='35px', margin='5px 0')

# Título Principal
encabezado = widgets.HTML(
    "<div style='background-color: #6A1B1B; color: white; text-align: center; padding: 10px; font-size: 20px; font-weight: bold; border-radius: 5px; font-family: sans-serif;'>"
    "Árbol de Recomendación Musical"
    "</div>"
)

# Panel Izquierdo (Consola y Gráfica)
panel_izquierdo = widgets.VBox([
    widgets.HTML("<h3 style='border-bottom: 1px solid #ccc; font-family: sans-serif;'>Estado del Árbol</h3>"),
    salida_consola,
    widgets.HTML("<h4 style='font-family: sans-serif; margin-top:15px;'>Representación en Memoria:</h4>"),
    salida_grafica
], layout=widgets.Layout(width='55%', padding='10px'))

# Panel Derecho (Botones y Entradas)
b1 = widgets.Button(description="1. Calcular tiempo total", button_style='info', layout=estilo_btn)
b1.on_click(btn1_calcular_click)

in_recomendar = widgets.Text(placeholder="Segundos...", layout=widgets.Layout(width='30%'))
b2 = widgets.Button(description="2. Recomendar canción", button_style='success', layout=widgets.Layout(width='65%'))
b2.on_click(btn2_recomendar_click)
fila2 = widgets.HBox([in_recomendar, b2], layout=estilo_btn)

in_filtrar = widgets.Text(placeholder="Min segundos...", layout=widgets.Layout(width='30%'))
b3 = widgets.Button(description="3. Filtrar canciones cortas", button_style='warning', layout=widgets.Layout(width='65%'))
b3.on_click(btn3_filtrar_click)
fila3 = widgets.HBox([in_filtrar, b3], layout=estilo_btn)

b4 = widgets.Button(description="4. Promedio de duración (propia)", button_style='primary', layout=estilo_btn)
b4.on_click(btn4_promedio_click)

in_insertar_tiempo = widgets.Text(placeholder="Tiempo (s)", layout=widgets.Layout(width='25%'))
in_insertar_titulo = widgets.Text(placeholder="Título", layout=widgets.Layout(width='25%'))
b5 = widgets.Button(description="5. Insertar canción", button_style='info', layout=widgets.Layout(width='45%'))
b5.on_click(btn5_insertar_click)
fila5 = widgets.HBox([in_insertar_tiempo, in_insertar_titulo, b5], layout=estilo_btn)

in_eliminar = widgets.Text(placeholder="Eliminar tiempo (s)", layout=widgets.Layout(width='40%'))
btn_elim = widgets.Button(description="Eliminar Nodo Específico", button_style='danger', layout=widgets.Layout(width='50%'))
btn_elim.on_click(btn_eliminar_click)
fila_elim = widgets.HBox([in_eliminar, btn_elim], layout=estilo_btn)

b6 = widgets.Button(description="6. Reiniciar con ejemplo", button_style='danger', layout=estilo_btn)
b6.on_click(btn6_reiniciar_click)

panel_derecho = widgets.VBox([
    widgets.HTML("<h3 style='border-bottom: 1px solid #ccc; font-family: sans-serif;'>Operaciones</h3>"),
    b1, fila2, fila3, b4, fila5, fila_elim, b6
], layout=widgets.Layout(width='45%', padding='10px', background_color='#f7f7f7'))

# Contenedor Principal
interfaz_final = widgets.VBox([
    encabezado,
    widgets.HBox([panel_izquierdo, panel_derecho])
], layout=widgets.Layout(width='100%', border='1px solid #ddd', padding='10px'))

display(interfaz_final)
btn6_reiniciar_click(None)